# سيرفر الصوتين الخفيفين على Hugging Face

يسوّي (أو يحدّث) الـSpace الخاص `omran77/omran-voices-tts`: يرفع ملفاته، ويحط فيه السر `HF_TOKEN`، وينتظر لين يشتغل،
ويقول جملة بكل صوت ويرفعها في `omran77/omran-voices-piper` داخل `samples/` (`space_m.wav` و`space_f.wav`) تسمعها.
موقع الأصوات بعدها يرسل له طلبات v5 وv4 بالعربي.

**على Kaggle:** Settings ← Accelerator **None** (ما يحتاج كرت) و Internet **On**، و Add-ons ← Secrets ← `HF_TOKEN` (✓)،
ثم Save Version ← **Save & Run All (Commit)**. حوالي ١٥ دقيقة.

In [ ]:
import os, time
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except ImportError:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
HF_TOKEN = (HF_TOKEN or '').strip()
assert HF_TOKEN.startswith('hf_'), 'السر HF_TOKEN ناقص'
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
print('Hugging Face account:', api.whoami()['name'])
SPACE = 'omran77/omran-voices-tts'
MODEL_REPO = 'omran77/omran-voices-piper'
Path('space').mkdir(exist_ok=True)

In [ ]:
%%writefile space/README.md
---
title: Omran Voices TTS
emoji: 🗣️
colorFrom: green
colorTo: blue
sdk: docker
app_port: 7860
pinned: false
---

The two light voices of omran-voices (Piper, formal Arabic): `POST /tts {"text", "voice": "m"|"f"}` → WAV.
Private: requests need the owner's Hugging Face token. Source: `space/` in the omran-voices repository.

In [ ]:
%%writefile space/Dockerfile
FROM python:3.11-slim
RUN useradd -m -u 1000 user
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
USER user
ENV HF_HOME=/home/user/.cache/huggingface
EXPOSE 7860
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "7860"]

In [ ]:
%%writefile space/requirements.txt
piper-tts==1.8.0
fastapi==0.115.12
uvicorn==0.34.2
huggingface_hub>=0.30,<1

In [ ]:
%%writefile space/app.py
"""The two light voices (Piper, formal Arabic) as a small web service, for a free CPU Space on Hugging Face.

  POST /tts   {"text": "...", "voice": "m" | "f"}  ->  audio/wav (22 050 Hz mono)
  GET  /health                                     ->  {"voices": ["f", "m"], "model": {...}}
  POST /reload                                     ->  loads the newest models (after a training run)

The Space is private: Hugging Face only lets in requests that carry a token of the owner
(Authorization: Bearer hf_...), so the service needs no key of its own. The models come from the
private repo omran77/omran-voices-piper with the Space's secret HF_TOKEN (read access is enough).
MODEL_DIR (a local folder with the same files) replaces the repo for local tests.
"""
import io
import os
import threading
import wave
from pathlib import Path

from fastapi import FastAPI, HTTPException
from fastapi.responses import Response
from pydantic import BaseModel

REPO = os.environ.get('MODEL_REPO', 'omran77/omran-voices-piper')
FILES = {'m': 'ar_AE-omran_m_fusha-medium.onnx', 'f': 'ar_AE-omran_f_fusha-medium.onnx'}
MAX_CHARS = 1200  # the app sends pieces of ~260 characters; this only stops abuse

voices, loaded = {}, {}
lock = threading.Lock()


def fetch(name):
    local = os.environ.get('MODEL_DIR')
    if local:
        return str(Path(local) / name)
    from huggingface_hub import hf_hub_download
    return hf_hub_download(REPO, name, token=os.environ.get('HF_TOKEN') or None, force_download=False)


def load():
    from piper import PiperVoice
    fresh, info = {}, {}
    for spk, name in FILES.items():
        model = fetch(name)
        fetch(name + '.json')
        fresh[spk] = PiperVoice.load(model)  # Arabic: Piper adds the diacritics itself (as in training)
        info[spk] = {'file': name, 'bytes': Path(model).stat().st_size, 'modified': int(Path(model).stat().st_mtime)}
    with lock:
        voices.clear()
        voices.update(fresh)
        loaded.clear()
        loaded.update(info)


app = FastAPI()


@app.on_event('startup')
def startup():
    load()


class Speak(BaseModel):
    text: str
    voice: str


@app.post('/tts')
def tts(req: Speak):
    text = ' '.join(req.text.split())
    if not text:
        raise HTTPException(400, 'empty text')
    if len(text) > MAX_CHARS:
        raise HTTPException(400, f'text longer than {MAX_CHARS} characters')
    with lock:
        voice = voices.get(req.voice)
    if voice is None:
        raise HTTPException(404, 'unknown voice: ' + req.voice)
    buf = io.BytesIO()
    with wave.open(buf, 'wb') as w:
        voice.synthesize_wav(text, w)
    return Response(buf.getvalue(), media_type='audio/wav', headers={'Cache-Control': 'no-store'})


@app.get('/health')
def health():
    with lock:
        return {'voices': sorted(voices), 'model': dict(loaded)}


@app.post('/reload')
def reload():
    # newest files from the repo (hf_hub_download fetches a file again when the repo has a newer one)
    load()
    return health()

In [ ]:
# the private Space with these files and the token it reads the models with
api.create_repo(SPACE, repo_type='space', space_sdk='docker', private=True, exist_ok=True)
assert api.space_info(SPACE).private, SPACE + ' لازم يكون Private'
api.add_space_secret(SPACE, 'HF_TOKEN', HF_TOKEN)
api.upload_folder(folder_path='space', repo_id=SPACE, repo_type='space', commit_message='omran voices tts')
print('uploaded to', SPACE)

In [ ]:
# building the image and loading the two models takes a few minutes
start, stage = time.time(), ''
while time.time() - start < 25 * 60:
    stage = str(api.get_space_runtime(SPACE).stage)
    print(time.strftime('%H:%M:%S'), stage)
    if stage.endswith('RUNNING'):
        break
    if 'ERROR' in stage:
        raise RuntimeError('the Space failed: open https://huggingface.co/spaces/' + SPACE + ' -> Logs')
    time.sleep(30)
assert stage.endswith('RUNNING'), 'the Space did not start in 25 minutes'

In [ ]:
# one sentence in each voice, through the same door the site uses (the private Space wants the token)
import requests
url = 'https://' + SPACE.replace('/', '-') + '.hf.space'
auth = {'Authorization': 'Bearer ' + HF_TOKEN}
for _ in range(20):  # the models may still be loading right after RUNNING
    h = requests.get(url + '/health', headers=auth, timeout=60)
    if h.ok and h.json().get('voices') == ['f', 'm']:
        break
    time.sleep(15)
print('health:', h.status_code, h.text[:300])
Path('out').mkdir(exist_ok=True)
for spk in ('m', 'f'):
    t = time.time()
    r = requests.post(url + '/tts', headers=auth, timeout=120,
                      json={'text': 'سنلتقي غدا إن شاء الله في البيت بعد صلاة المغرب.', 'voice': spk})
    print(spk, r.status_code, r.headers.get('content-type'), len(r.content), 'bytes in', round(time.time() - t, 1), 's')
    assert r.ok and r.content[:4] == b'RIFF', r.text[:300]
    Path(f'out/space_{spk}.wav').write_bytes(r.content)
api.upload_folder(folder_path='out', path_in_repo='samples', repo_id=MODEL_REPO, commit_message='space test')
print('listen: https://huggingface.co/' + MODEL_REPO + '/tree/main/samples  (space_m.wav, space_f.wav)')